# Aula 02 · Apache Spark: processamento distribuído

## O que é
Imagine que você precisa **corrigir mil provas**. Sozinho, levaria dias. Com 10 professores, cada um corrige 100 e o trabalho termina muito mais rápido. Um coordenador distribui as provas e junta as notas no final.

O Spark funciona exatamente assim:

| Na correção de provas | No Spark |
|---|---|
| Coordenador | **Driver**: planeja e distribui o trabalho |
| Professores | **Executores**: fazem o trabalho em paralelo |
| Pilhas de provas | **Partições**: pedaços dos dados |
| A planilha de notas | **DataFrame**: os dados organizados em linhas e colunas |
| "Separe as provas de matemática" | **Transformação**: descreve o que fazer, ainda sem fazer |
| "Me diga a média agora" | **Ação**: dispara o trabalho de verdade |

O coordenador só começa quando alguém pede um resultado. Isso se chama **avaliação preguiçosa (lazy evaluation)**: o Spark junta todas as transformações e monta o plano mais eficiente antes de executar.

## Por que importa
O Spark executa praticamente todas as cargas de trabalho do Databricks. Entender como ele trabalha explica **o tempo e o custo** de cada job que vamos orquestrar.

## Como aplicar

In [ ]:
caminho = "/Volumes/growdev/bronze/landing/loans"
df_loans = spark.read.parquet(caminho)

# Ler já é uma transformação "barata": o Spark só olhou o schema dos arquivos
df_loans.printSchema()

In [ ]:
from pyspark.sql import functions as F

df_resultado = (
    df_loans
    .filter(F.lower(F.col("status")) == "ativo")   # transformação
    .groupBy("term_months")                        # transformação
    .count()                                       # transformação (em DataFrame, count após groupBy é transformação)
    .orderBy("term_months")                        # transformação
)
print("Repare: esta célula terminou na hora. Nenhum dado foi processado ainda.")

In [ ]:
# explain mostra o plano de execução que o driver montou
df_resultado.explain()

In [ ]:
# display é uma ação: agora sim os executores trabalham
display(df_resultado)

### Observando a execução
Depois de rodar a célula acima, clique em **See performance** logo abaixo do resultado. Você vê as etapas que o Spark executou e quanto tempo cada uma levou. No compute serverless essa tela substitui a antiga Spark UI.

In [ ]:
# spark_partition_id mostra em qual partição (pilha de provas) cada linha está
display(
    df_loans.withColumn("particao", F.spark_partition_id())
    .groupBy("particao").count()
    .orderBy("particao")
)

## Checklist da aula
- [ ] Sei explicar driver, executores e partições com a analogia das provas
- [ ] Sei a diferença entre transformação e ação
- [ ] Vi que nada roda até eu pedir um resultado (lazy evaluation)
- [ ] Encontrei o plano de execução e a tela de performance